# Compare FES2014 to the tides that come with CoDEC

In [ ]:
### %matplotlib widget

import pathlib

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import gcvalid.util.constants as u_const
import gcvalid.util.gauge as u_gauge


CODEC_2019_DIR = pathlib.Path("/p/projects/tumble/mengel/isimip/20211220_CoDEC_data_Idai/")
CODEC_2019_FILE = CODEC_2019_DIR / "idai_tides_his.nc"

# Pilots Station East, S.W. Pass, LA - Station ID: 8760922
# https://tidesandcurrents.noaa.gov/stationhome.html?id=8760922
new_orleans_coord = (-89.4067, 28.93167)
katrina_period = (np.datetime64("2005-08-01"), np.datetime64("2005-08-31"))
katrina_period = (np.datetime64("2005-01-01T00"), np.datetime64("2005-12-31T23"))
katrina_map = "2005236N23285-0"

# Wilmington, NC - Station ID: 8658120
# https://tidesandcurrents.noaa.gov/stationhome.html?id=8658120
wilmington_coord = (-77.953, 34.227)
florence_period = (np.datetime64("2018-09-12"), np.datetime64("2018-09-18"))
florence_period = (np.datetime64("2018-01-01T00"), np.datetime64("2018-12-31T23"))
florence_map = "2018242N13343-0"

beira_coord = (34.878, -19.849)
idai_period = (np.datetime64("2019-03-01"), np.datetime64("2019-03-31"))
idai_map = None

STATION_COORD = wilmington_coord
PERIOD = florence_period
FLOODMAP = florence_map

In [ ]:
if PERIOD[1] <= np.datetime64("2018-12-31T23:59"):
    ds = xr.open_dataset(u_const.CODEC_COORD_FILE)
    dx = ds.station_x_coordinate.values - STATION_COORD[0]
    dy = ds.station_y_coordinate.values - STATION_COORD[1]
    station_i = np.argmin(dx**2 + dy**2)
    station_filename = f"gtsm_station{station_i:05d}.nc"
    ds_tides = xr.open_dataset(u_const.CODEC_TIDES_DIR / station_filename)
    ds_tides = ds_tides.rename_dims(index="time").rename_vars(index="time")
else:
    # load from 2019 CoDEC dataset (for Idai)
    ds_tides = xr.open_dataset(CODEC_2019_FILE)
    dx = ds_tides.station_x_coordinate.values - STATION_COORD[0]
    dy = ds_tides.station_y_coordinate.values - STATION_COORD[1]
    station_i = np.argmin(dx**2 + dy**2)
    ds_tides = ds_tides.isel(stations=station_i)
    # convert m to mm
    ds_tides['waterlevel'] *= 1000

t_mask = (
    (ds_tides.time >= PERIOD[0])
    & (ds_tides.time <= PERIOD[1])
)
ds_tides = ds_tides.sel(time=t_mask)

In [ ]:
ds_tides["fes"] = ("time", u_gauge.compute_fes_tides(
    *STATION_COORD,
    ds_tides.time.values.astype('datetime64[us]'),
))

In [ ]:
if FLOODMAP is not None:
    gaugedata = u_gauge.load_gaugedata("gfd", FLOODMAP, referenced=True, by_gsrc=False, filter_gsrc=["gesla3"])
    locs = np.array([stdata["location"] for stdata in gaugedata])
    station_i = np.argmin((
        (locs - np.array(STATION_COORD)[::-1])**2
    ).sum(axis=-1))
    stdata = gaugedata[station_i]
    print(stdata["filename"])
    times = ds_tides.time.values
    mm_conv_factor = u_const.GAUGE_SOURCE_MM_CONVERSION["gesla3"]
    for harmonics_suffix in ["", "_90d"]:
        key = f'harmonics{harmonics_suffix}'
        tide_fun = u_gauge.harmonics_fun(stdata[key])
        ds_tides[key] = ("time", tide_fun(times) * mm_conv_factor - stdata["annual_msl"])

In [ ]:
ds_tides = ds_tides[["waterlevel", "fes"] + (
    [] if FLOODMAP is None else ["harmonics", "harmonics_90d"]
)].resample(time="1d").mean()
s_wlevel = None if FLOODMAP is None else (
    stdata["combined"].resample("1d").mean() - stdata["annual_msl"]
)

In [ ]:
fig = plt.figure(figsize=(16, 4))
ax = fig.add_subplot(1, 1, 1)
ax.plot(ds_tides.time, ds_tides.waterlevel, label="CoDEC (tides_his)")
ax.plot(ds_tides.time, ds_tides.fes, label="FES2014")
if FLOODMAP is not None:
    ax.plot(ds_tides.time, ds_tides.harmonics, label="Harmonics estimate")
    ax.plot(ds_tides.time, ds_tides.harmonics_90d, label="Harmonics 90d estimate")
    ax.plot(s_wlevel.index, s_wlevel.values, label="Gesla3")
    ax.set_ylim(-300, 1000)
ax.legend()
ax.set_ylabel("Water level (mm)")
ax.set_xlabel("Date")
fig.tight_layout()